In [ ]:
Section 2: Model Development with scikit-learn

For both datasets, regression models were selected because the target variables are numerical.  
Dataset 1 predicts student grades, Dataset 2 predicts GPA change.

Each dataset uses:
- train/test split
- preprocessing pipeline
- one-hot encoding for categorical features
- standard scaling for numerical features
- multiple regression models for comparison

In [1]:
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
import pandas as pd

In [3]:
# Dataset 1: Gaming Dataset
df1 = pd.read_csv("Gaming_Academic_Performance.csv")
#DropID
df1 = df1.drop(columns=["student_id"])
#Features and Target
X1 = df1.drop(columns=["grades"])
y1 = df1["grades"]
# Identify feature types
numeric_features_1 = X1.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features_1 = X1.select_dtypes(include=["object"]).columns.tolist()
# Preprocessing
numeric_transformer_1 = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="mean")),
    ("scaler", StandardScaler())
])
categorical_transformer_1 = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])
preprocessor_1 = ColumnTransformer(transformers=[
    ("num", numeric_transformer_1, numeric_features_1),
    ("cat", categorical_transformer_1, categorical_features_1)
])
# Train/test split
X1_train, X1_test, y1_train, y1_test = train_test_split(
    X1, y1, test_size=0.2, random_state=42
)
# Models
models_1 = {
    "Linear Regression": LinearRegression(),
    "KNN Regressor": KNeighborsRegressor(n_neighbors=5),
    "Random Forest Regressor": RandomForestRegressor(random_state=42)
}
results_1 = []

for name, model in models_1.items():
    pipeline = Pipeline(steps=[
        ("preprocessor", preprocessor_1),
        ("model", model)
    ])
    
    pipeline.fit(X1_train, y1_train)
    y1_pred = pipeline.predict(X1_test)
    
    mae = mean_absolute_error(y1_test, y1_pred)
    rmse = np.sqrt(mean_squared_error(y1_test, y1_pred))
    r2 = r2_score(y1_test, y1_pred)
    
    results_1.append([name, mae, rmse, r2])

results_1_df = pd.DataFrame(results_1, columns=["Model", "MAE", "RMSE", "R2"])
display(results_1_df.sort_values(by="R2", ascending=False))

,Model,MAE,RMSE,R2
2,Random Forest Regressor,4.885081,6.307026,0.920641
0,Linear Regression,5.503880,6.965510,0.903205
1,KNN Regressor,6.903789,8.698109,0.849063


In [ ]:
Dataset 1 Model Choice:
Linear Regression was the baseline model since its simple and interpretable.
KNN Regressor because it can capture the local patterns between students who were similar.
Random Forest Regressor because it can model nonlinear relationships and multiple features.
Using the train/test split to evaluate each model.

In [5]:
# Dataset 2: AI Student Life Data
df2 = pd.read_csv("AI_Impact_Student_Life_2026.csv")
#target variable
df2["GPA_Change"] = df2["GPA_Post_AI"] - df2["GPA_Baseline"]
# DropID
df2 = df2.drop(columns=["Student_ID"])
#leakage columns
X2 = df2.drop(columns=["GPA_Change", "GPA_Post_AI", "GPA_Baseline"])
y2 = df2["GPA_Change"]
#types
numeric_features_2 = X2.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features_2 = X2.select_dtypes(include=["object"]).columns.tolist()
#Preprocessing
numeric_transformer_2 = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="mean")),
    ("scaler", StandardScaler())
])
categorical_transformer_2 = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])
preprocessor_2 = ColumnTransformer(transformers=[
    ("num", numeric_transformer_2, numeric_features_2),
    ("cat", categorical_transformer_2, categorical_features_2)
])
#Train/test split
X2_train, X2_test, y2_train, y2_test = train_test_split(
    X2, y2, test_size=0.2, random_state=42
)
#Models
models_2 = {
    "Linear Regression": LinearRegression(),
    "Decision Tree Regressor": DecisionTreeRegressor(random_state=42),
    "Random Forest Regressor": RandomForestRegressor(random_state=42)
}
results_2 = []

for name, model in models_2.items():
    pipeline = Pipeline(steps=[
        ("preprocessor", preprocessor_2),
        ("model", model)
    ])
    
    pipeline.fit(X2_train, y2_train)
    y2_pred = pipeline.predict(X2_test)
    
    mae = mean_absolute_error(y2_test, y2_pred)
    rmse = np.sqrt(mean_squared_error(y2_test, y2_pred))
    r2 = r2_score(y2_test, y2_pred)
    
    results_2.append([name, mae, rmse, r2])

results_2_df = pd.DataFrame(results_2, columns=["Model", "MAE", "RMSE", "R2"])
display(results_2_df.sort_values(by="R2", ascending=False))

,Model,MAE,RMSE,R2
0,Linear Regression,0.118876,0.138552,-0.025141
2,Random Forest Regressor,0.119527,0.140338,-0.051745
1,Decision Tree Regressor,0.162300,0.197682,-1.086877


In [ ]:
Dataset 2 Model Choice:
Linear Regression was the baseline model since target variable is numerical.
Decision Tree Regressor because it can capture decision-based patterns in the ai usage.
Random Forest Regressor because it can help imporve the decision model by averaging the tress.
Using the train/test split to evaluate each model.

In [10]:
#Just to see
# Cross-validation for Dataset 1
cv_model_1 = Pipeline(steps=[
    ("preprocessor", preprocessor_1),
    ("model", RandomForestRegressor(random_state=42))
])
cv_scores_1 = cross_val_score(cv_model_1, X1, y1, cv=5, scoring="r2")
print("Dataset 1 Random Forest CV R2 scores:", cv_scores_1)
print("Dataset 1 Average CV R2:", cv_scores_1.mean())
# Cross-validation for Dataset 2
cv_model_2 = Pipeline(steps=[
    ("preprocessor", preprocessor_2),
    ("model", RandomForestRegressor(random_state=42))
])
cv_scores_2 = cross_val_score(cv_model_2, X2, y2, cv=5, scoring="r2")
print("Dataset 2 Random Forest CV R2 scores:", cv_scores_2)
print("Dataset 2 Average CV R2:", cv_scores_2.mean())

Dataset 1 Random Forest CV R2 scores: [0.92418802 0.92555022 0.93199975 0.92872821 0.92822416]
Dataset 1 Average CV R2: 0.9277380717408976
Dataset 2 Random Forest CV R2 scores: [-0.07871025 -0.08516416 -0.09519098 -0.02395607 -0.08208353]
Dataset 2 Average CV R2: -0.07302099707780255


In [ ]:
Model Development with scikit-learn
Dataset 1: Gaming/Student Performance
For the first dataset, three regression models were used to predict the target variable, 
which were linear regression, K-Nearest Neighbors (KNN) regressor, and random forest regressor.
The models were choosen for the baseline model, a distance based modelm, and ensemble tree based model.

The dataset was split in train and test sets and using 80/20 split. Numerical features were scaled using
Standard Scaler, and categorical variables were encoded using OneHot for the preprocessing.

The Results showed the Random forest regressor performed best, and got a R^2 score of 0.9206, wiht lower
error among the three models (MAE = 4.8851, RMSE = 6.3070). The relectionship between gaming behavior,
lifestyle factors, and academic performance is likely nonlinear, and the rando forest model captured these
patterns better.

Dataset 2: AI Student Life
For dataset two, the regression models were tested to predit the GPA change, which is linear regression,
decision tree regressor, and random forest regressor.

To avoid data leakage, the columns for post ai and baseline were removed from the input features
after creating the target variable. The dataset was also split using an 80/20 train/test split. Numerical'
features were scaled, while categorical were one-hot encoded.

Among the models, linear regression performed the best, with R^2 of -0.0251, MAE of 0.1189, and RMSE of 0.1386.
The performance is weak and it is still the best of the three. The negative value across all the models suggest that the
available features may not strongly predict GPA change. This may be since GPA chnages were small,
noisy, and influenced by many external factors not captured in the dataset.

Model Selection Justification:
The models were chosen to provide a range of machine learning:
    
Linear regression was used as a simple and interpretable baseline.
KNN regressor was used to test whether local similarity.
Decision tree regressor was used to capture rule-based nonlinear relationships.
Random Forest Regressor was used as a stronger ensemble model capable of handling complex feature interactions.

This comparison showed that the model performance depends heavily on the dataset.
The gaming dataset had a stronger predictive structure than the AI student life dataset.